# Missing-data investigation

**Historical investigation, not current terrain coverage.** The original Earth tile/halo defect was corrected in [notebook 09](09_verify_earth_coverage_fix.ipynb). Lunar source heights were finite; our image-support mask introduced the elevation holes. [Notebook 13](13_build_lunar_terrain_views.ipynb) supplies a full-context view with zero elevation gaps inside the seven ROIs. Read [notebook 14](../exploration/14_terrain_feature_eda.ipynb) for the corrected terrain and separate support views. Remaining derivative edge exclusions and genuine environmental missingness are different issues.

Investigate the five pinned completed runs before patch extraction. Raw and processed inputs are read-only. Diagnostic GeoTIFFs, maps, CSVs and findings are written to a new folder under `notebooks/diagnostics/`.

The Diviner check independently scans raw fixed-width projected coordinates for every product with an empty region. It reads about 27 GB and may take several minutes. It does not rerun notebook 06 or change its exports. Coordinate rounding is handled with a one-metre boundary band.

Terrain checks distinguish source nodata, image support, mask coverage and neighborhood exclusions. Climate checks compare native-cell missingness with high-resolution land/water fractions. Results distinguish expected exclusion from source coverage limits and possible defects; they do not establish that new observations are recoverable.

Cells can run from the project root or a folder under notebooks using the existing project environment. For this investigation they are executed serially through project Python, with saved outputs; no notebooks 02-06 are executed.

**Workflow stage: preprocessing / data readiness.** Use this notebook to inspect coverage, investigate exclusions, or verify corrections before EDA. Its saved results and pinned inputs preserve the original investigation. Notebook 13 builds the two lunar terrain views; notebook 14 explores the corrected Earth run and those views. See the [workflow and acceptance checks](../README.md). Moving this notebook does not change when the original findings were made.


In [ ]:
import json
import sys
import xml.etree.ElementTree as ET
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.warp import reproject
from rasterio.windows import from_bounds
from scipy.ndimage import convolve, label

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'outline.md').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
from processing import AREAS, MOON_CRS, Grid, roi_mask, warp

RUNS = {
    'earth': ROOT / 'data/processed/earth_terrain_landcover/20261028T014314564317Z_8a3e5d',
    'moon': ROOT / 'data/processed/lunar_terrain_environment/20261028T022855014566Z_cffea7',
    'climate': ROOT / 'data/processed/earth_climate/20261028T014917410405Z_d7c8d5',
    'diviner': ROOT / 'data/processed/diviner_temperature/20261028T022942126792Z_d44299',
}
OUT = ROOT / 'notebooks/diagnostics' / ('missing-data-' + datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True, exist_ok=False)
results = {'runs': {k: str(v.relative_to(ROOT)) for k, v in RUNS.items()}}
evidence = []


def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


def raster(path):
    with rasterio.open(path) as src:
        return src.read(1, masked=True).astype('float32').filled(np.nan), Grid(src.crs, src.transform, src.width, src.height)


def aggregate(binary, source, grid):
    destination = np.full((grid.height, grid.width), np.nan, dtype='float32')
    reproject(binary.astype('float32'), destination, src_transform=source.transform, src_crs=source.crs,
              dst_transform=grid.transform, dst_crs=grid.crs, dst_nodata=np.nan, resampling=Resampling.average)
    return destination


def neighborhoods(valid, size):
    return convolve(valid.astype('int16'), np.ones((size, size), dtype='int16'), mode='constant', cval=0) == size * size


def save_categories(name, values, grid, legend):
    path = OUT / (name + '.tif')
    with rasterio.open(path, 'w', driver='GTiff', width=grid.width, height=grid.height, count=1,
                       dtype='uint8', transform=grid.transform, crs=grid.crs, compress='deflate') as dst:
        dst.write(values.astype('uint8'), 1)
        dst.update_tags(categories=json.dumps(legend), purpose='missing-data diagnosis; not a model feature')
    colors = np.array([[90,90,90], [38,166,100], [230,170,40], [120,75,190],
                       [220,55,70], [45,145,220], [250,70,220], [240,240,240]], dtype='uint8')
    with rasterio.open(path) as src:
        h = max(1, round(500 * src.height / max(src.width, src.height)))
        w = max(1, round(500 * src.width / max(src.width, src.height)))
        small = src.read(1, out_shape=(h, w))
    rgb = colors[small].transpose(2,0,1)
    with rasterio.open(OUT / (name + '.png'), 'w', driver='PNG', width=w, height=h, count=3, dtype='uint8') as dst:
        dst.write(rgb)
    return {'file': path.name, 'preview': name + '.png', 'legend': legend}


def checkpoint():
    (OUT / 'findings.json').write_text(json.dumps(results, indent=2, allow_nan=False), encoding='utf-8')


for path in RUNS.values():
    assert read_json(path/'provenance.json')['status'] == 'complete'
print('Diagnostic outputs:', OUT, flush=True)

In [ ]:
# DIVINER: independently scan raw projected x/y fields, not the production lon/lat transform.
summary = pd.read_csv(RUNS['diviner']/'regional_bin_summary.csv')
empty = summary[summary.valid_temperature_rows == 0]
region_bounds = read_json(RUNS['diviner']/'provenance.json')['config']['regions']
ns = {'p': 'http://pds.nasa.gov/pds4/pds/v1'}
diviner_rows = []
raw_base = ROOT / 'data/raw/moon/diviner/seasonal'
for number, product in enumerate(sorted(empty.source_product.unique()), 1):
    xml_path = raw_base / (product + '.xml')
    root = ET.parse(xml_path).getroot()
    table = root.find('.//p:Table_Character', ns)
    assert table is not None
    count = int(table.findtext('p:records', namespaces=ns))
    offset = int(table.findtext('p:offset', namespaces=ns))
    record_length = int(table.findtext('.//p:record_length', namespaces=ns))
    fields = {f.findtext('p:name', namespaces=ns): f for f in table.findall('.//p:Field_Character', ns)}
    selected = ['x', 'y', 'tbol']
    dtype = np.dtype({'names': selected,
                      'formats': ['S' + fields[k].findtext('p:field_length', namespaces=ns) for k in selected],
                      'offsets': [int(fields[k].findtext('p:field_location', namespaces=ns))-1 for k in selected],
                      'itemsize': record_length})
    raw_path = raw_base / (product + '.tab')
    assert raw_path.stat().st_size == offset + count * record_length
    missing_codes = {k: [float(c.text) for c in fields[k].findall('p:Special_Constants/*', ns)] for k in selected}
    # Source x/y have six decimal places, normalized by 1,737,400 m: rounding <=0.869 m per axis.
    x_description = fields['x'].findtext('p:description', namespaces=ns) or ''
    assert 'normalized' in x_description.lower()
    targets = empty[empty.source_product == product].region.tolist()
    counters = {region: {'interior_rows': 0, 'interior_valid_temperature': 0, 'boundary_band_rows': 0} for region in targets}
    data = np.memmap(raw_path, dtype=dtype, mode='r', offset=offset, shape=(count,))
    for start in range(0, count, 500_000):
        chunk = data[start:start+500_000]
        xraw, yraw = chunk['x'].astype('float64'), chunk['y'].astype('float64')
        known = np.isfinite(xraw) & np.isfinite(yraw) & ~np.isin(xraw, missing_codes['x']) & ~np.isin(yraw, missing_codes['y'])
        x, y = xraw * 1737400, yraw * 1737400
        for region in targets:
            west,south,east,north = region_bounds[region]
            inner = known & (x >= west+1) & (x < east-1) & (y >= south+1) & (y < north-1)
            expanded = known & (x >= west-1) & (x < east+1) & (y >= south-1) & (y < north+1)
            counters[region]['interior_rows'] += int(inner.sum())
            counters[region]['boundary_band_rows'] += int((expanded & ~inner).sum())
            temperatures = chunk['tbol'][inner].astype('float64')
            valid = np.isfinite(temperatures) & (temperatures > 0) & ~np.isin(temperatures, missing_codes['tbol'])
            counters[region]['interior_valid_temperature'] += int(valid.sum())
    del chunk, data
    for region in targets:
        values = counters[region]
        classification = ('source_omits_rows' if values['interior_rows']==0 and values['boundary_band_rows']==0
                          else 'investigate_selection' if values['interior_valid_temperature']>0
                          else 'boundary_or_temperature_validity')
        diviner_rows.append({'region': region, 'source_product': product, **values, 'classification': classification})
    if number % 10 == 0:
        print('Diviner raw products checked:', number, '/', empty.source_product.nunique(), flush=True)
        pd.DataFrame(diviner_rows).to_csv(OUT/'diviner_source_gap_checks.csv',index=False)
results['diviner'] = {'empty_pairs':len(empty), 'source_products_scanned':int(empty.source_product.nunique()),
                      'classification_counts':pd.Series([r['classification'] for r in diviner_rows]).value_counts().to_dict()}
pd.DataFrame(diviner_rows).to_csv(OUT/'diviner_source_gap_checks.csv',index=False)
heat = summary.copy()
heat['empty'] = heat.valid_temperature_rows == 0
heat.pivot(index='region',columns='source_product',values='empty').astype('uint8').to_csv(OUT/'diviner_missing_matrix.csv')
checkpoint()
print('Diviner result', results['diviner'], flush=True)

In [ ]:
# LUNAR: partition raw missingness and processing exclusions at both native and output grids.
lunar_rows = []
maps = []
for region, bounds in region_bounds.items():
    directory = ROOT/'data/raw/moon/sdem/regions'/region
    elevation_path = next(directory.glob('*_GLDELEV_005.tif'))
    mask_path = next(directory.glob('*_GLDMASK_005.tif'))
    raw_z, source_grid = raster(elevation_path)
    counts = warp(mask_path, source_grid, Resampling.nearest)
    with rasterio.open(mask_path) as mask:
        mask_shape, mask_resolution, mask_bounds = mask.shape, mask.res, tuple(mask.bounds)
    # All observed files are unrotated same-CRS grids. Check this explicitly before using bounds.
    assert source_grid.transform.b == source_grid.transform.d == 0
    with rasterio.open(mask_path) as mask:
        assert mask.crs == source_grid.crs and mask.transform.b == mask.transform.d == 0
    xs = source_grid.transform.c + (np.arange(source_grid.width)+.5)*source_grid.transform.a
    ys = source_grid.transform.f + (np.arange(source_grid.height)+.5)*source_grid.transform.e
    in_mask = ((ys[:,None] >= mask_bounds[1]) & (ys[:,None] < mask_bounds[3]) &
               (xs[None,:] >= mask_bounds[0]) & (xs[None,:] < mask_bounds[2]))
    valid_z = np.isfinite(raw_z)
    supported = valid_z & np.isfinite(counts) & (counts>0)
    folder = RUNS['moon']/'moon'/region/'grid_30m'
    z, grid = raster(folder/'elevation_m.tif')
    recorded_support, _ = raster(folder/'image_support_fraction.tif')
    inside = roi_mask(grid, bounds, MOON_CRS)
    expected_support = aggregate(supported, source_grid, grid)
    expected_support[~inside] = np.nan
    raw_reasons = [~valid_z, valid_z & ~in_mask, valid_z & in_mask & ~np.isfinite(counts),
                   valid_z & np.isfinite(counts) & (counts<=0)]
    fractions = [aggregate(reason,source_grid,grid) for reason in raw_reasons]
    dominant = np.argmax(np.nan_to_num(np.stack(fractions)),axis=0)+2
    joint = np.isfinite(z)
    mismatches = {}
    for layer,size in [('slope_deg',3),('laplacian_per_m',3),('local_elevation_std_m',11)]:
        values,_ = raster(folder/(layer+'.tif'))
        expected = neighborhoods(np.isfinite(z),size)
        mismatches[layer] = int((np.isfinite(values)!=expected).sum())
        joint &= np.isfinite(values)
    relative,_=raster(folder/'relative_elevation_m.tif')
    joint &= np.isfinite(relative)
    expected_z = inside & (expected_support>=.99)
    finite_z = np.isfinite(z)
    categories = np.zeros(z.shape,dtype='uint8')
    categories[inside & ~finite_z] = dominant[inside & ~finite_z]
    categories[finite_z] = 6  # Derived features absent, reassigned below when all present.
    categories[joint] = 1
    mapinfo=save_categories(region+'_gap_causes',categories,grid,
        {0:'outside ROI',1:'all five terrain channels valid',2:'source elevation nodata dominates',
         3:'mask extent missing dominates',4:'mask nodata dominates',5:'zero image support dominates',6:'neighborhood feature exclusion'})
    maps.append(mapinfo)
    row={'region':region,'raw_elevation_cells':int(raw_z.size),'raw_elevation_missing':int((~valid_z).sum()),
         'mask_shape':str(mask_shape),'mask_resolution_m':str(mask_resolution),
         'raw_mask_outside_extent':int((~in_mask).sum()),'raw_mask_nodata_inside_extent':int((in_mask & ~np.isfinite(counts)).sum()),
         'raw_zero_image_support':int((np.isfinite(counts)&(counts<=0)).sum()),
         'support_fraction_mismatches':int((~np.isclose(recorded_support,expected_support,atol=1e-6,equal_nan=True)).sum()),
         'elevation_validity_mismatches':int((finite_z!=expected_z).sum()),'feature_validity_mismatches':mismatches,
         'roi_cells':int(inside.sum()),'processed_elevation_missing_inside':int((inside & ~finite_z).sum()),
         'neighborhood_exclusions':int((finite_z & ~joint).sum()),
         'neighborhood_exclusions_touch_roi_edge':int((finite_z & ~joint & ~neighborhoods(inside,11)).sum()),
         'neighborhood_exclusions_internal':int((finite_z & ~joint & neighborhoods(inside,11)).sum()),
         'joint_feature_cells':int(joint.sum())}
    lunar_rows.append(row)
    print('Lunar causes:', region, row['elevation_validity_mismatches'], 'unexplained elevation cells',flush=True)
    del raw_z,counts,valid_z,supported,in_mask,raw_reasons,fractions
results['lunar']=lunar_rows
pd.DataFrame(lunar_rows).to_csv(OUT/'lunar_gap_causes.csv',index=False)
results['maps']=maps
checkpoint()

In [ ]:
# EARTH: distinguish neighborhood exclusions at the ROI boundary from internal missing-neighbor exclusions.
earth_rows=[]
for region, info in AREAS.items():
    folder=RUNS['earth']/'earth'/region/'grid_30m'
    z,grid=raster(folder/'elevation_m.tif')
    inside=roi_mask(grid,info['bbox'])
    finite=np.isfinite(z)
    natural,_=raster(folder/'natural_screen_mask.tif')
    expected=warp(folder.parent/'native_mosaic.tif',grid)
    expected[~inside]=np.nan
    joint=finite.copy(); mismatches={}
    for layer,size in [('slope_deg',3),('laplacian_per_m',3),('local_elevation_std_m',11)]:
        a,_=raster(folder/(layer+'.tif'))
        mismatches[layer]=int((np.isfinite(a)!=neighborhoods(finite,size)).sum())
        joint &= np.isfinite(a)
    relative,_=raster(folder/'relative_elevation_m.tif');joint &= np.isfinite(relative)
    missing=(natural==1)&~joint
    boundary=missing&~neighborhoods(inside,11)
    internal=missing&neighborhoods(inside,11)
    raw_missing=inside&~finite
    components,count=label(raw_missing)
    component_sizes=np.bincount(components.ravel())[1:]
    # Read the native mosaic at each missing output cell center as an independent source check.
    rows,cols=np.where(raw_missing)
    xs=grid.transform.c+(cols+.5)*grid.transform.a; ys=grid.transform.f+(rows+.5)*grid.transform.e
    from rasterio.warp import transform
    with rasterio.open(folder.parent/'native_mosaic.tif') as mosaic:
        gx,gy=transform(grid.crs,mosaic.crs,xs.tolist(),ys.tolist()) if len(xs) else ([],[])
        samples=np.array([s[0] for s in mosaic.sample(zip(gx,gy),masked=True)]) if len(xs) else np.array([])
        sampled_missing=int((~np.isfinite(samples)|(samples==mosaic.nodata)).sum())
    categories=np.zeros(z.shape,dtype='uint8')
    categories[inside]=7;categories[(natural==1)&joint]=1;categories[boundary]=2;categories[internal]=3;categories[raw_missing]=4
    maps.append(save_categories(region+'_screen_gap_causes',categories,grid,
        {0:'outside ROI',1:'screened with complete features',2:'screened: ROI boundary neighborhood',
         3:'screened: internal missing-neighbor exclusion',4:'elevation missing inside ROI',7:'not selected by natural screen'}))
    earth_rows.append({'region':region,'screened_missing_features':int(missing.sum()),'roi_boundary_exclusions':int(boundary.sum()),
                       'internal_neighborhood_exclusions':int(internal.sum()),'elevation_missing_inside_roi':int(raw_missing.sum()),
                       'missing_elevation_with_native_missing_or_outside_at_center':sampled_missing,
                       'source_warp_validity_mismatches':int((finite!=np.isfinite(expected)).sum()),
                       'feature_validity_mismatches':mismatches,'elevation_gap_components':int(count),
                       'largest_elevation_gap_cells':int(component_sizes.max()) if count else 0})
results['earth']=earth_rows
pd.DataFrame(earth_rows).to_csv(OUT/'earth_gap_causes.csv',index=False)
checkpoint()
print('Earth gap attribution:',earth_rows,flush=True)

In [ ]:
# CLIMATE: compare all Lanzarote missing cells with WorldCover land/water support.
base=RUNS['climate']/'earth/lanzarote/native_monthly'
first=pd.read_csv(base/'tmin_2015.csv.gz')
first=first[first.date=='2015-01-01'].sort_values(['latitude','longitude'],ascending=[False,True])
lon=np.sort(first.longitude.unique());lat=np.sort(first.latitude.unique())[::-1]
dx=float(np.median(np.diff(lon)));dy=float(-np.median(np.diff(lat)))
grid=Grid('EPSG:4326',from_origin(lon[0]-dx/2,lat[0]+dy/2,dx,dy),len(lon),len(lat))
known_climate=first.value.notna().to_numpy().reshape(len(lat),len(lon))
changes=0
for p in sorted(base.glob('*.csv.gz')):
    frame=pd.read_csv(p)
    for _,group in frame.groupby('date'):
        ordered=group.sort_values(['latitude','longitude'],ascending=[False,True])
        assert np.array_equal(ordered[['longitude','latitude']].to_numpy(),first[['longitude','latitude']].to_numpy())
        changes+=int((ordered.value.notna().to_numpy().reshape(known_climate.shape)!=known_climate).sum())
cover=ROOT/'data/raw/earth/worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N27W015_Map.tif'
bounds=(grid.transform.c,grid.transform.f-grid.height*dy,grid.transform.c+grid.width*dx,grid.transform.f)
with rasterio.open(cover) as src:
    win=from_bounds(*bounds,transform=src.transform).round_offsets().round_lengths()
    codes=src.read(1,window=win)
    source_grid=Grid(src.crs,src.window_transform(win),codes.shape[1],codes.shape[0])
    source_known=np.isin(codes,[10,20,30,40,50,60,70,80,90,95,100])
    land=source_known&(codes!=80)
    land_fraction=aggregate(land,source_grid,grid)
    water_fraction=aggregate(source_known&(codes==80),source_grid,grid)
    known_fraction=aggregate(source_known,source_grid,grid)
weights=np.cos(np.deg2rad(lat))[:,None]
total_land=float((land_fraction*weights).sum())
represented_land=float((land_fraction*weights*known_climate).sum())
categories=np.zeros(known_climate.shape,dtype='uint8')
categories[known_climate]=1
categories[~known_climate&(land_fraction<.01)]=2
categories[~known_climate&(land_fraction>=.01)&(land_fraction<.5)]=3
categories[~known_climate&(land_fraction>=.5)]=4
maps.append(save_categories('lanzarote_climate_land_support',categories,grid,
    {0:'unclassified',1:'climate cell available',2:'missing climate; <1% mapped land',
     3:'missing climate; 1-50% mapped land',4:'missing climate; >=50% mapped land'}))
frame=first[['longitude','latitude']].copy()
frame['climate_available']=known_climate.ravel();frame['worldcover_land_fraction']=land_fraction.ravel()
frame['worldcover_water_fraction']=water_fraction.ravel();frame['worldcover_known_fraction']=known_fraction.ravel()
frame.to_csv(OUT/'lanzarote_climate_land_support.csv',index=False)
results['climate']={'grid_cells':int(known_climate.size),'available_cells':int(known_climate.sum()),
    'missing_cells':int((~known_climate).sum()),'availability_changes_across_600_month_variable_tables':changes,
    'missing_cells_below_1pct_land':int((categories==2).sum()),'missing_cells_1_to_50pct_land':int((categories==3).sum()),
    'missing_cells_at_least_50pct_land':int((categories==4).sum()),
    'mapped_land_weight_in_available_climate_cells_fraction':represented_land/total_land,
    'worldcover_known_fraction_min':float(known_fraction.min()),
    'scope':'Area-weighted approximation within complete footprints of selected climate cells, not the exact configured box; land includes all known WorldCover classes except permanent water. This measures coarse-grid representation, not independent climate observations.'}
results['maps']=maps
checkpoint()
print('Lanzarote land support:',results['climate'],flush=True)

In [ ]:
# Positive controls ensure the independent Diviner coordinate check can find real observations.
summary = pd.read_csv(RUNS['diviner']/'regional_bin_summary.csv')
region_bounds = read_json(RUNS['diviner']/'provenance.json')['config']['regions']
control_products = summary[summary.valid_temperature_rows > 0].groupby('region').first().source_product
controls = []
ns = {'p': 'http://pds.nasa.gov/pds4/pds/v1'}
for product in sorted(control_products.unique()):
    root = ET.parse(ROOT/'data/raw/moon/diviner/seasonal'/f'{product}.xml').getroot()
    table = root.find('.//p:Table_Character',ns)
    fields = {f.findtext('p:name',namespaces=ns):f for f in table.findall('.//p:Field_Character',ns)}
    dtype = np.dtype({'names':['x','y'],
        'formats':['S'+fields[k].findtext('p:field_length',namespaces=ns) for k in ['x','y']],
        'offsets':[int(fields[k].findtext('p:field_location',namespaces=ns))-1 for k in ['x','y']],
        'itemsize':int(table.findtext('.//p:record_length',namespaces=ns))})
    raw = np.memmap(ROOT/'data/raw/moon/diviner/seasonal'/f'{product}.tab',dtype=dtype,mode='r',
                    offset=int(table.findtext('p:offset',namespaces=ns)),shape=(int(table.findtext('p:records',namespaces=ns)),))
    x,y=raw['x'].astype('float64')*1737400,raw['y'].astype('float64')*1737400
    for region in control_products[control_products==product].index:
        w,s,e,n=region_bounds[region]
        inner=(x>=w+1)&(x<e-1)&(y>=s+1)&(y<n-1)
        outer=(x>=w-1)&(x<e+1)&(y>=s-1)&(y<n+1)
        count=int(summary[(summary.region==region)&(summary.source_product==product)].iloc[0].selected_rows)
        assert int(inner.sum())<=count<=int(outer.sum()) and count>0
        controls.append({'region':region,'source_product':product,'raw_definite_rows':int(inner.sum()),
                         'raw_possible_rows':int(outer.sum()),'exported_rows':count})
    del raw,x,y
pd.DataFrame(controls).to_csv(OUT/'diviner_positive_controls.csv',index=False)
results['diviner']['positive_controls']=controls

# Locate Arizona's internal missing-neighbor exclusions against independently supplied land cover.
folder=RUNS['earth']/'earth/northern_arizona/grid_30m'
z,grid=raster(folder/'elevation_m.tif')
inside=roi_mask(grid,AREAS['northern_arizona']['bbox'])
holes=inside&~np.isfinite(z)
water,_=raster(folder/'water_fraction.tif')
natural,_=raster(folder/'natural_screen_mask.tif')
cover=ROOT/'data/raw/earth/worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N33W114_Map.tif'
rows,cols=np.where(holes)
xs=grid.transform.c+(cols+.5)*grid.transform.a;ys=grid.transform.f+(rows+.5)*grid.transform.e
from rasterio.warp import transform

with rasterio.open(cover) as src:
    lon,lat=transform(grid.crs,src.crs,xs.tolist(),ys.tolist())
    codes=np.array([sample[0] for sample in src.sample(zip(lon,lat))])
labels,number=label(holes)
components=[]
for component in range(1,number+1):
    selected=labels[rows,cols]==component
    components.append({'component':component,'cells':int(selected.sum()),
                       'center_longitude':float(np.mean(np.asarray(lon)[selected])),
                       'center_latitude':float(np.mean(np.asarray(lat)[selected])),
                       'water_class_centers':int((codes[selected]==80).sum())})
pd.DataFrame(components).sort_values('cells',ascending=False).to_csv(OUT/'arizona_source_gap_locations.csv',index=False)
results['arizona_mosaic_gaps_landcover']={'cells':int(holes.sum()),'worldcover_center_classes':{str(int(k)):int(v) for k,v in zip(*np.unique(codes,return_counts=True))},
                                'cells_with_any_water_fraction':int((water[holes]>0).sum()),
                                'cells_with_majority_water_fraction':int((water[holes]>=.5).sum()),
                                'cells_selected_by_natural_screen':int((natural[holes]==1).sum()),
                                'interpretation':'Land-cover correspondence identifies gap locations, not their cause; the next cell distinguishes source extents from true nodata.'}
checkpoint()
print('Diviner positive controls:',controls,flush=True)
print('Arizona hole land cover:',results['arizona_mosaic_gaps_landcover'],flush=True)


In [ ]:
# Separate gaps outside the selected tiles from nodata inside a selected tile.
import shutil
import tempfile
import zipfile

from rasterio.warp import transform

folder = RUNS['earth']/'earth/northern_arizona'
z,grid = raster(folder/'grid_30m/elevation_m.tif')
inside = roi_mask(grid,AREAS['northern_arizona']['bbox'])
rows,cols = np.where(inside & ~np.isfinite(z))
xs = grid.transform.c+(cols+.5)*grid.transform.a
ys = grid.transform.f+(rows+.5)*grid.transform.e
lon,lat = transform(grid.crs,'EPSG:4326',xs.tolist(),ys.tolist())
lon,lat = np.asarray(lon),np.asarray(lat)
covered = np.zeros(len(rows),dtype=bool)
for path in sorted((folder/'extracted').glob('*.tif')):
    with rasterio.open(path) as src:
        covered |= (lon>=src.bounds.left)&(lon<src.bounds.right)&(lat>=src.bounds.bottom)&(lat<src.bounds.top)
archive = ROOT/'data/raw/earth/fabdem/archives/N30W120-N40W110_FABDEM_V1-2.zip'
neighbors = ['N34W113_FABDEM_V1-2.tif','N34W112_FABDEM_V1-2.tif',
             'N35W111_FABDEM_V1-2.tif','N34W111_FABDEM_V1-2.tif']
available = np.zeros(len(rows),dtype=bool)
tile_results = []
with tempfile.TemporaryDirectory(prefix='neighbor-control-',dir=OUT) as temporary, zipfile.ZipFile(archive) as zipped:
    for name in neighbors:
        path = Path(temporary)/name
        with zipped.open(name) as source, path.open('wb') as destination:
            shutil.copyfileobj(source,destination)
        with rasterio.open(path) as src:
            selected = (lon>=src.bounds.left)&(lon<src.bounds.right)&(lat>=src.bounds.bottom)&(lat<src.bounds.top)
            indices = np.flatnonzero(selected)
            samples = np.array([v.filled(np.nan)[0] for v in src.sample(zip(lon[selected],lat[selected]),masked=True)])
            available[indices] |= np.isfinite(samples)
            tile_results.append({'archive_member':name,'missing_centers_covered':int(selected.sum()),
                                 'finite_source_samples':int(np.isfinite(samples).sum()),
                                 'member_crc32':zipped.getinfo(name).CRC})
results['arizona_tile_coverage'] = {'missing_output_cells':len(rows),
    'inside_selected_tile_extents':int(covered.sum()),
    'outside_selected_tile_extents':int((~covered).sum()),
    'finite_neighbor_samples':int(available.sum()),'neighbor_tiles':tile_results,
    'classification':'Recoverable tile-selection coverage loss. Neighbor sampling proves source support at all counted centers, not that a full bilinear/terrain rerun has been performed.'}
for row in results['earth']:
    old='missing_elevation_with_native_nodata_at_center'
    if old in row:
        row['missing_elevation_with_native_missing_or_outside_at_center']=row.pop(old)
if 'arizona_source_holes' in results:
    results['arizona_mosaic_gaps_landcover']=results.pop('arizona_source_holes')
pd.DataFrame(results['earth']).to_csv(OUT/'earth_gap_causes.csv',index=False)
pd.DataFrame(tile_results).to_csv(OUT/'arizona_neighbor_tile_support.csv',index=False)
results['status']='running'
checkpoint()
print('Arizona tile coverage:',results['arizona_tile_coverage'],flush=True)


In [ ]:
# Validate method oracles and complete the diagnostic evidence record.
import hashlib
import importlib.metadata
import platform

import nbformat

# Analytical fixtures: full 5x5 occupancy has exactly nine full 3x3 neighborhoods.
known = np.ones((5, 5), dtype=bool)
expected = np.zeros((5, 5), dtype=bool)
expected[1:4, 1:4] = True
np.testing.assert_array_equal(neighborhoods(known, 3), expected)
source = Grid(MOON_CRS, from_origin(0, 15, 5, 5), 3, 3)
target = Grid(MOON_CRS, from_origin(0, 15, 15, 15), 1, 1)
occupancy = np.ones((3, 3), dtype=bool)
occupancy[1, 1] = False
np.testing.assert_allclose(aggregate(occupancy, source, target), [[8/9]], atol=1e-7)

# Verify the scan really covered every expected month/variable and matching pinned settings.
climate_config = read_json(RUNS['climate']/'provenance.json')['config']
expected_keys = {(v, f'{y}-{m:02d}-01') for v in climate_config['variables']
                 for y in climate_config['years'] for m in range(1, 13)}
observed_keys = []
for path in sorted((RUNS['climate']/'earth/lanzarote/native_monthly').glob('*.csv.gz')):
    frame = pd.read_csv(path)
    observed_keys.extend(frame[['variable','date']].drop_duplicates().itertuples(index=False, name=None))
assert len(observed_keys) == len(set(observed_keys))
assert set(observed_keys) == expected_keys
results['climate']['month_variable_tables_checked'] = len(observed_keys)
earth_config = read_json(RUNS['earth']/'provenance.json')['config']
lunar_config = read_json(RUNS['moon']/'provenance.json')['config']
assert json.loads(json.dumps(AREAS)) == earth_config['areas']  # JSON arrays versus Python tuples
assert earth_config['roughness_window_cells'] == lunar_config['roughness_window'] == 11
assert lunar_config['minimum_image_support_fraction'] == .99
assert lunar_config['moon_radius_m'] == 1737400

# Hash every source actually used for diagnosis, plus the completed derived evidence.
inputs = {ROOT/'notebooks/processing.py', ROOT/'data/raw/earth/fabdem/archives/N30W120-N40W110_FABDEM_V1-2.zip'}
inputs.update((RUNS['earth']/'earth/northern_arizona/extracted').glob('*.tif'))
for run in RUNS.values():
    inputs.add(run/'provenance.json')
products = set(pd.read_csv(OUT/'diviner_source_gap_checks.csv').source_product) | set(pd.read_csv(OUT/'diviner_positive_controls.csv').source_product)
for product in products:
    inputs.update((ROOT/'data/raw/moon/diviner/seasonal'/f'{product}{suffix}') for suffix in ['.tab','.xml'])
for region in read_json(RUNS['diviner']/'provenance.json')['config']['regions']:
    directory = ROOT/'data/raw/moon/sdem/regions'/region
    inputs.update(directory.glob('*_GLDELEV_005.tif'))
    inputs.update(directory.glob('*_GLDMASK_005.tif'))
    inputs.add(directory/'README.txt')
inputs.add(ROOT/'data/raw/earth/worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N27W015_Map.tif')
inputs.add(ROOT/'data/raw/earth/worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N33W114_Map.tif')
for stage in ['earth','moon']:
    inputs.update(RUNS[stage].glob('*/*/grid_30m/*.tif'))
inputs.update(RUNS['earth'].glob('earth/*/native_mosaic.tif'))
inputs.update((RUNS['climate']/'earth/lanzarote/native_monthly').glob('*.csv.gz'))
inputs.add(RUNS['diviner']/'regional_bin_summary.csv')
input_records = []
for number, path in enumerate(sorted(inputs),1):
    h = hashlib.sha256()
    before = path.stat()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            h.update(chunk)
    after = path.stat()
    assert (before.st_size,before.st_mtime_ns) == (after.st_size,after.st_mtime_ns)
    input_records.append({'path':str(path.relative_to(ROOT)),'bytes':after.st_size,
                          'mtime_ns':after.st_mtime_ns,'sha256':h.hexdigest()})
    if number % 50 == 0:
        print('Diagnostic source hashes:',number,'/',len(inputs),flush=True)
notebook = nbformat.read(ROOT/'notebooks/preprocessing/08_investigate_missing_data.ipynb',as_version=4)
source_text = '\n\n'.join(c.source for c in notebook.cells if c.cell_type=='code')
provenance = {'python':platform.python_version(),
              'packages':{k:importlib.metadata.version(k) for k in ['numpy','pandas','rasterio','scipy','nbformat']},
              'notebook_code_sha256':hashlib.sha256(source_text.encode()).hexdigest(),
              'effective_areas':AREAS,'effective_moon_crs':MOON_CRS,
              'minimum_image_support_fraction':.99,'roughness_window_cells':11,
              'coordinate_boundary_tolerance_m':1,'inputs':input_records,
              'scope':'Raw and processed sources read-only; evidence hashes captured after diagnostic scans. File stability was checked during hashing; hashes establish this inspected evidence snapshot, not historical immutability.'}
(OUT/'diagnostic_provenance.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')
results['status'] = 'complete'
results['completed_utc'] = datetime.now(UTC).isoformat()
results['method_oracles'] = 'Known occupancy and aggregation cases passed; complete climate key set checked.'
checkpoint()
print('Missing-data investigation complete:',OUT,flush=True)


## Findings from the completed investigation

- **Confirmed recoverable coverage loss:** all 5,986 missing Arizona elevation centres fall outside the selected tile footprints. All have finite elevations in neighboring tiles already downloaded. No corrected Earth processing run has been performed.
- **Earth feature boundaries:** 63,423 screened cells lacking features touch the ROI edge; 2,587 others surround Arizona coverage gaps. A valid processing halo may recover boundary features.
- **Lunar terrain:** no source elevation nodata; the gaps match image-support exclusions and valid-neighborhood requirements.
- **Diviner:** all 243 empty region/product pairs have no raw observations in the region or the one-metre boundary band. Seven positive controls match exactly.
- **Lanzarote climate:** all 115 missing cells have <1% mapped land. Available cells contain approximately 99.95% of the mapped land within selected native-cell footprints.

[Full report](../diagnostics/missing-data-20261028T054352Z/REPORT.md) · [Map gallery](../diagnostics/missing-data-20261028T054352Z/gallery.html)

The next correction is Earth tile coverage and processing halos, before patch selection. Raw and processed inputs remain unchanged.